# Phishing URL Classification Demo

COSC 50523 — Deep Minds
Leiton Peterson

Adapted from the team's week4/C4_gradio_demo.py script.

This demo loads the saved default character CNN and predicts a class
from pasted URL text. It does not visit the website or retrain the model.

The output is a model prediction, not a guarantee that a website is safe.
The model processes at most the first 200 characters of a URL.

In [1]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/DeepMinds')
sys.path.insert(0, str(DATA_DIR))

required_files = [
    'project_utils.py',
    'torch_models.py',
    'char_cnn_default.pt',
]

for filename in required_files:
    file_path = DATA_DIR / filename
    assert file_path.is_file(), f"Missing file: {file_path}"
    print("Found:", filename)

Mounted at /content/drive
Found: project_utils.py
Found: torch_models.py
Found: char_cnn_default.pt


In [2]:
%pip install -q gradio

In [3]:
import torch
import gradio as gr

from project_utils import encode_urls, strip_prefix
from torch_models import CharCNN, predict_proba, DEVICE

checkpoint = torch.load(
    DATA_DIR / 'char_cnn_default.pt',
    map_location='cpu',
    weights_only=True,
)

cfg = checkpoint['config']
vocab = checkpoint['vocab']

model = CharCNN(
    vocab_size=len(vocab) + 2,
    embed_dim=cfg['embed_dim'],
    filters=cfg['filters'],
    kernel_sizes=cfg['kernel_sizes'],
    dropout=cfg['dropout'],
    hidden=cfg['hidden'],
    use_conv=cfg['use_conv'],
).to(DEVICE)

model.load_state_dict(checkpoint['state_dict'])
model.eval()

print("Saved CNN loaded successfully.")
print("Device:", DEVICE)
print("Maximum input length:", cfg['max_len'])
print("Gradio version:", gr.__version__)

Saved CNN loaded successfully.
Device: cpu
Maximum input length: 200
Gradio version: 6.31.0


In [4]:
def check_url(url):
    url = url.strip()

    if not url:
        raise gr.Error("Please paste a URL first.")

    # Match the preprocessing used when this model was trained.
    if cfg.get('strip_prefix', False):
        url = strip_prefix(url)

    if len(url) > cfg['max_len']:
        gr.Warning(
            f"Only the first {cfg['max_len']} characters will be processed."
        )

    encoded = encode_urls([url], vocab, cfg['max_len'])

    probability = float(
        predict_proba(model, encoded, x_dtype=torch.long)[0]
    )

    return {
        'Phishing': probability,
        'Legitimate': 1 - probability,
    }


demo = gr.Interface(
    fn=check_url,
    inputs=gr.Textbox(
        label="URL",
        placeholder="https://example.com",
        lines=2,
    ),
    outputs=gr.Label(
        label="Model scores",
        num_top_classes=2,
    ),
    title="Deep Minds: Phishing URL Demo",
    description=(
        "Paste a complete URL, including http:// or https://. "
        "The saved default CNN analyzes the text without visiting the website. "
        "Its scores are model estimates, not a guarantee of website safety."
    ),
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d7d86507a3e4e981cd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Running the demo

Run the notebook cells in order and authorize access to the shared
DeepMinds Drive folder. The saved CNN checkpoint and shared Python
helpers are required; no dataset loading or model training is needed.

Paste a complete URL and click Submit. The model analyzes URL text
without visiting the website. Inputs longer than the saved model's
200-character limit are truncated.

The Gradio share link is temporary and requires the Colab session
to remain running. Rerun the notebook to launch a new session.

## Limitations

The interface displays model scores, not verified website classifications
or guarantees of safety. Scores can be highly confident and incorrect.
This demo uses the default CNN checkpoint from notebook 06, not
necessarily the configuration selected for final evaluation.